<a href="https://colab.research.google.com/github/NehaMyageri04/FLAD/blob/main/VQC_circuit_on_case_5_MPAF_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/NehaMyageri04/FLAD.git

Cloning into 'FLAD'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 86 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 13.36 MiB | 19.09 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [ ]:
%cd FLAD/1.FLAD

/content/FLAD/1.FLAD


In [ ]:
!pip install torch torchvision scikit-learn

In [ ]:
!python main.py

Extracting ../data/MNIST/train-images-idx3-ubyte.gz
Extracting ../data/MNIST/train-labels-idx1-ubyte.gz
Extracting ../data/MNIST/t10k-images-idx3-ubyte.gz
Extracting ../data/MNIST/t10k-labels-idx1-ubyte.gz

communicate round 1
honest feature in server dataset: [0.37629411 0.96977603]
eps: 7.566409970943853e-05
label of all clients: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1]
label: 0, mean: [0.37627122 0.96973938]
cos: 0.9999999980842594, length: 4.081574854486458e-05
label score: [(np.int64(0), np.float64(0.4999795911678573))]
malicious clients: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
accuracy: 0.9360001087188721

communicate round 2
honest feature in server dataset: [0.45939073 0.96497804]
eps: 8.12139751283046e-05
label of all clients: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 -1 -1 -1 -1 -1 -1 

In [ ]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 39.1 MB/s eta 0:00:00


In [ ]:
import pennylane as qml
print(qml.__version__)

0.44.1


In [ ]:
import pennylane as qml
import numpy as np

dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def circuit(x):
    qml.RY(x[0], wires=0)
    qml.RY(x[1], wires=1)
    return qml.expval(qml.PauliZ(0))

print(circuit([0.5, 0.2]))

0.8775825618903726


In [ ]:
import pennylane as qml
import numpy as np
from pennylane import numpy as pnp

# ── Setup ──────────────────────────────────────
n_qubits = 4
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def vqc_circuit(features, weights):
    qml.AngleEmbedding(features, wires=range(n_qubits), rotation='X')
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

# ── Fake Data (simulating gradient features) ───
# Clean client updates → small, normal values
clean_samples = np.array([
    [0.1, 0.2, 0.15, 0.1],
    [0.2, 0.1, 0.2,  0.15],
    [0.1, 0.3, 0.1,  0.2],
    [0.15,0.1, 0.25, 0.1],
])

# Poisoned client updates → large, abnormal values
poison_samples = np.array([
    [2.5, 3.1, 2.8, 3.0],
    [3.0, 2.9, 3.2, 2.7],
    [2.8, 3.0, 2.9, 3.1],
    [3.1, 2.8, 3.0, 2.9],
])

# Labels: +1 = clean, -1 = malicious
X = np.vstack([clean_samples, poison_samples])
y = np.array([1, 1, 1, 1, -1, -1, -1, -1], dtype=float)

print("✅ Data ready")
print(f"Clean samples: {len(clean_samples)}")
print(f"Poison samples: {len(poison_samples)}")
print(f"Feature shape: {X.shape}")

✅ Data ready
Clean samples: 4
Poison samples: 4
Feature shape: (8, 4)


In [ ]:
# ── Initialize weights ─────────────────────────
np.random.seed(42)
weights = pnp.random.randn(n_layers, n_qubits, 3, requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.1)

# ── Normalize features to [0, pi] ─────────────
def normalize(x):
    x = np.array(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min() + 1e-8) * np.pi

# ── Cost Function ──────────────────────────────
def cost(weights):
    loss = 0.0
    for xi, yi in zip(X, y):
        xi_norm = normalize(xi)
        pred = vqc_circuit(xi_norm, weights)
        loss += (pred - yi) ** 2
    return loss / len(X)

# ── Training Loop (FIXED) ──────────────────────
print("Training VQC...")
for epoch in range(30):
    weights, loss_val = opt.step_and_cost(cost, weights)  # ← FIXED LINE
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Loss: {float(loss_val):.4f}")

print("\n✅ Training complete")

Training VQC...
Epoch   0 | Loss: 1.2782
Epoch   5 | Loss: 0.9465
Epoch  10 | Loss: 0.8415
Epoch  15 | Loss: 0.7199
Epoch  20 | Loss: 0.6685
Epoch  25 | Loss: 0.5539

✅ Training complete


In [ ]:
# ── New clean samples with bigger separation ───
clean_samples = np.array([
    [0.05, 0.08, 0.06, 0.07],
    [0.10, 0.06, 0.09, 0.08],
    [0.07, 0.09, 0.05, 0.10],
    [0.06, 0.07, 0.08, 0.05],
])

poison_samples = np.array([
    [2.5, 3.1, 2.8, 3.0],
    [3.0, 2.9, 3.2, 2.7],
    [2.8, 3.0, 2.9, 3.1],
    [3.1, 2.8, 3.0, 2.9],
])

X = np.vstack([clean_samples, poison_samples])
y = np.array([1, 1, 1, 1, -1, -1, -1, -1], dtype=float)

# ── More qubits + layers ───────────────────────
n_qubits = 6   # ← was 4
n_layers = 3   # ← was 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def vqc_circuit(features, weights):
    qml.AngleEmbedding(features, wires=range(n_qubits), rotation='X')
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

def normalize(x):
    x = np.array(x, dtype=float)
    # Pad to n_qubits
    if len(x) < n_qubits:
        x = np.pad(x, (0, n_qubits - len(x)))
    return (x - x.min()) / (x.max() - x.min() + 1e-8) * np.pi

def cost(weights):
    loss = 0.0
    for xi, yi in zip(X, y):
        xi_norm = normalize(xi)
        pred = vqc_circuit(xi_norm, weights)
        loss += (pred - yi) ** 2
    return loss / len(X)

# ── Fresh weights + higher LR ─────────────────
np.random.seed(0)
weights = pnp.random.randn(n_layers, n_qubits, 3, requires_grad=True)
opt = qml.AdamOptimizer(stepsize=0.15)  # ← higher LR

# ── Train ──────────────────────────────────────
print("Training VQC (v3)...")
for epoch in range(150):
    weights, loss_val = opt.step_and_cost(cost, weights)
    if epoch % 15 == 0:
        print(f"Epoch {epoch:3d} | Loss: {float(loss_val):.4f}")

print("\n✅ Training complete")

# ── Test ───────────────────────────────────────
print("\n🔍 VQC Detection Results (v3):")
print("-" * 40)
correct = 0
for i, (xi, yi) in enumerate(zip(X, y)):
    xi_norm = normalize(xi)
    output = float(vqc_circuit(xi_norm, weights))
    predicted = 1 if output > 0 else -1
    actual =   "Clean ✅"    if yi == 1  else "Poisoned ❌"
    detected = "Clean ✅"    if predicted == 1 else "Poisoned ❌"
    status =   "✓" if predicted == yi else "✗ WRONG"
    print(f"Client {i:2d} | Actual: {actual} | VQC says: {detected} | {status}")
    if predicted == yi:
        correct += 1

print("-" * 40)
print(f"\nVQC Accuracy: {correct}/{len(X)} = {correct/len(X)*100:.1f}%")

Training VQC (v3)...
Epoch   0 | Loss: 0.9839
Epoch  15 | Loss: 0.5639
Epoch  30 | Loss: 0.4714
Epoch  45 | Loss: 0.4411
Epoch  60 | Loss: 0.4361
Epoch  75 | Loss: 0.4350
Epoch  90 | Loss: 0.4348
Epoch 105 | Loss: 0.4347
Epoch 120 | Loss: 0.4347
Epoch 135 | Loss: 0.4347

✅ Training complete

🔍 VQC Detection Results (v3):
----------------------------------------
Client  0 | Actual: Clean ✅ | VQC says: Clean ✅ | ✓
Client  1 | Actual: Clean ✅ | VQC says: Clean ✅ | ✓
Client  2 | Actual: Clean ✅ | VQC says: Clean ✅ | ✓
Client  3 | Actual: Clean ✅ | VQC says: Clean ✅ | ✓
Client  4 | Actual: Poisoned ❌ | VQC says: Poisoned ❌ | ✓
Client  5 | Actual: Poisoned ❌ | VQC says: Poisoned ❌ | ✓
Client  6 | Actual: Poisoned ❌ | VQC says: Poisoned ❌ | ✓
Client  7 | Actual: Poisoned ❌ | VQC says: Poisoned ❌ | ✓
----------------------------------------

VQC Accuracy: 8/8 = 100.0%


In [ ]:
import sys
sys.path.append("FLAD/1.FLAD")

In [ ]:
%cd FLAD/1.FLAD
!python main.py

[Errno 2] No such file or directory: 'FLAD/1.FLAD'
/content/FLAD/1.FLAD
Extracting ../data/MNIST/train-images-idx3-ubyte.gz
Extracting ../data/MNIST/train-labels-idx1-ubyte.gz
Extracting ../data/MNIST/t10k-images-idx3-ubyte.gz
Extracting ../data/MNIST/t10k-labels-idx1-ubyte.gz

communicate round 1
honest feature in server dataset: [0.39166114 0.96779507]
eps: 0.00026351036164153936
label of all clients: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 -1 -1 -1 -1 -1 -1 -1 -1
 -1 -1]
label: 0, mean: [0.39167617 0.96778097]
cos: 0.9999999979551396, length: 7.118678081563523e-06
label score: [(np.int64(0), np.float64(0.499996439638529))]
DBSCAN malicious clients: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[VQC] Bootstrap round 1/5 — collecting training data...
[VQC] Using DBSCAN result this round (bootstrap phase)
accuracy: 0.9515997171401978

communicate round 2
honest feature in server dataset: [0.67218512 0.96342671